# Baseline de churn — Regressão Logística

Este notebook registra o baseline de Regressão Logística. O pré-processamento fica dentro do pipeline e a validação cruzada usa somente dados de treino. **Nota retrospectiva:** a avaliação de teste executada originalmente já foi consultada e repetida; seus números são históricos e exploratórios, não uma estimativa independente. Para preservar essa separação, não reexecute nem use esses números para escolher modelos. Uma avaliação final exige dados rotulados novos.

## 1. Carregar e preparar os dados

O caminho do CSV é relativo à pasta `notebooks`. Convertemos `TotalCharges` em número, removemos `customerID` por ser um identificador e codificamos `Churn` como 0 para `No` e 1 para `Yes`. Fixamos a semente para tornar as etapas aleatórias reproduzíveis.

In [1]:
RANDOM_STATE = 42

import json
import time
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(RANDOM_STATE)

df = pd.read_csv("../data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = df["Churn"].map({"No": 0, "Yes": 1})
if y.isna().any():
    raise ValueError("A coluna Churn contém valores fora das categorias No/Yes.")

X = df.drop(columns=["customerID", "Churn"])
print(f"Dimensões dos dados: {df.shape}")
print(f"Valores ausentes em TotalCharges após conversão: {X['TotalCharges'].isna().sum()}")
print(f"Clientes com churn: {int(y.sum())} ({y.mean():.2%})")
display(X.head())

Dimensões dos dados: (7043, 21)
Valores ausentes em TotalCharges após conversão: 11
Clientes com churn: 1869 (26.54%)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


## 2. Separação usada no experimento histórico

A divisão estratificada original reservou 20% para teste. Esse conjunto já foi usado em avaliações anteriores e não é mais cego. O pipeline e a validação cruzada abaixo usam somente a porção de treino. Não usamos os resultados históricos de teste para escolher hiperparâmetros ou modelos.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Partição de desenvolvimento usada no baseline histórico: {len(X_train)} linhas.")
print(f"Partição antiga reservada, não avaliada nesta reexecução: {len(X_test)} linhas.")

Partição de desenvolvimento usada no baseline histórico: 5634 linhas.
Partição antiga reservada, não avaliada nesta reexecução: 1409 linhas.


## 3. Pipeline e validação cruzada no treino

O `ColumnTransformer` imputa a mediana e padroniza as colunas numéricas. Para as colunas categóricas, aplica one-hot encoding e ignora categorias inéditas. Todas as transformações estão dentro do `Pipeline`, portanto são ajustadas separadamente em cada partição de treino da validação cruzada. Calculamos média e desvio-padrão de F1, AUC-ROC, precisão e recall para churn.

In [3]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, make_column_selector(dtype_include="number")),
        (
            "categorical",
            categorical_pipeline,
            make_column_selector(dtype_exclude="number"),
        ),
    ]
)

baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        ),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "f1": "f1",
    "auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
}

training_start = time.perf_counter()
cv_results = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise",
)

cv_metrics = {}
for metric in scoring:
    scores = cv_results[f"test_{metric}"]
    cv_metrics[f"{metric}_media"] = scores.mean()
    cv_metrics[f"{metric}_dp"] = scores.std(ddof=1)

cv_summary = pd.DataFrame(
    {
        "métrica": ["F1 (churn)", "AUC-ROC", "Precisão (churn)", "Recall (churn)"],
        "média": [cv_metrics["f1_media"], cv_metrics["auc_media"], cv_metrics["precision_media"], cv_metrics["recall_media"]],
        "desvio-padrão": [cv_metrics["f1_dp"], cv_metrics["auc_dp"], cv_metrics["precision_dp"], cv_metrics["recall_dp"]],
    }
)
display(cv_summary.round(4))

fit_start = time.perf_counter()
baseline.fit(X_train, y_train)
final_fit_seconds = time.perf_counter() - fit_start
training_seconds = time.perf_counter() - training_start
print(f"Pipeline ajustado no treino completo em {final_fit_seconds:.2f} s.")
print(f"Tempo total de validação cruzada e ajuste final: {training_seconds:.2f} s.")

Pipeline ajustado no treino completo em 0.05 s.
Tempo total de validação cruzada e ajuste final: 0.42 s.


,métrica,média,desvio-padrão
0,F1 (churn),0.5923,0.0331
1,AUC-ROC,0.8461,0.0140
2,Precisão (churn),0.6529,0.0285
3,Recall (churn),0.5431,0.0460


## 4. Resultados de teste históricos — somente exploratórios

Os valores abaixo são saídas preservadas da execução original e não serão recalculados neste notebook. O conjunto foi consultado e a avaliação foi repetida, portanto os resultados não devem ser apresentados como uma estimativa final independente nem ser usados para ajustar o modelo. O limiar original foi 0,5.

## 4. Resultados de teste históricos — somente exploratórios

Os valores abaixo são saídas preservadas da execução original e não serão recalculados neste notebook. O conjunto foi consultado e a avaliação foi repetida, portanto os resultados não devem ser apresentados como uma estimativa final independente nem ser usados para ajustar o modelo. O limiar original foi 0,5.

## 5. Referência majoritária histórica — somente exploratória

Na mesma avaliação histórica, o classificador que sempre previa não churn obteve acurácia 0,7346, mas F1, precisão e recall de churn iguais a 0. Esse resultado é apenas ilustrativo, pois usa o conjunto de teste já consultado.

## 5. Referência majoritária histórica — somente exploratória

Na mesma avaliação histórica, o classificador que sempre previa não churn obteve acurácia 0,7346, mas F1, precisão e recall de churn iguais a 0. Esse resultado é apenas ilustrativo, pois usa o conjunto de teste já consultado.

## 6. Registro histórico do experimento

A linha original do baseline foi preservada em `docs/experimentos.csv` e identificada como exploratória. Não atualize métricas de teste a partir deste notebook; para estimativas de generalização, use a validação aninhada no notebook comparativo e obtenha novos dados antes de uma avaliação prospectiva.

## 6. Registro histórico do experimento

A linha original do baseline foi preservada em `docs/experimentos.csv` e identificada como exploratória. Não atualize métricas de teste a partir deste notebook; para estimativas de generalização, use a validação aninhada no notebook comparativo e obtenha novos dados antes de uma avaliação prospectiva.

## 7. Conclusões e próximos passos

A validação cruzada abaixo é a referência histórica do baseline no conjunto de treino. Os números do teste vieram da execução original e foram consultados; por isso, ficam apenas para registro.

| Métrica | Validação cruzada no treino | Teste histórico (exploratório) |
|---|---:|---:|
| F1 (churn) | 0,5923 ± 0,0331 | 0,6040 |
| AUC-ROC | 0,8461 ± 0,0140 | 0,8419 |
| Precisão (churn) | 0,6529 ± 0,0285 | 0,6572 |
| Recall (churn) | 0,5431 ± 0,0460 | 0,5588 |

**Referência majoritária histórica:** o classificador que sempre previa não churn obteve acurácia 0,7346, mas F1, precisão e recall de churn iguais a zero. Não use essa avaliação como resultado independente.

**Pontos fracos:** a Regressão Logística é linear e pode não representar relações complexas. O limiar padrão pode deixar de identificar clientes que cancelam. Não há neste dataset um teste independente não consultado.

**Próximos passos:** comparar modelos usando validação cruzada aninhada e obter dados rotulados novos para uma avaliação final, após congelar o modelo e o limiar.